In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter

from qiskit_aer import AerSimulator


In [ ]:
def uniform_over_range(num_qubits: int, M: int):
    """
    Returns a circuit that prepares a uniform superposition over |0>,|1>,...,|M-1> on num_qubits qubits.
    Uses a Hadamard layer if M is a power of 2, else uses the method of Shukla and Vedula.
    """
    if M not in range(2 ** num_qubits +1):
        print(M)
        print(num_qubits)
        raise Exception('Bad M: out of range')
    for i in range(num_qubits+1):
        if M == 2 ** i:
            print(f'M={M} a power of 2. Use Hadamard circuit.')
            circuit = QuantumCircuit(num_qubits)
            for j in range(i):
                circuit.h(j)
                
            return circuit
    
    circuit = QuantumCircuit(num_qubits)

    try:
        M_binary = np.binary_repr(M, num_qubits)
    except Exception as e:
        print(M)
        print(num_qubits)
        raise e
    M_binary = M_binary[::-1]
    ran = np.arange(len(M_binary))
    mask = [M_binary[x] == '1' for x in range(len(M_binary))]
    l = ran[mask]
    
    for i in range(1, len(l)):
        circuit.x(l[i])
    if l[0] > 0:
        for i in range(l[0]):
            circuit.h(i)

    MM = 2 ** l[0]

    circuit.ry(-2 * np.arccos(np.sqrt(MM/M)), l[1])

    for i in range(l[0], l[1]):
        circuit.ch(l[1], i, ctrl_state=0)

    for m in range(1, len(l)-1):
        circuit.cry(
            -2 * np.arccos(np.sqrt(2 ** l[m] / (M - MM) )), 
            l[m], l[m+1], ctrl_state=0
        )
        for i in range(l[m], l[m+1]):
            circuit.ch(l[m+1], i, ctrl_state=0)
        MM += 2 ** l[m]

    return circuit


def state_prep(N: int, T: int) -> QuantumCircuit:
    n = int(np.ceil(np.log2(N+1)))
    uni = uniform_over_range(n, N+1)
    circuit = QuantumCircuit(n * T)
    for t in range(T):
        circuit.append(
            uni,
            list(range(t * n, (t+1) * n))   
        )
    return circuit


def get_mixer_operator(N: int, T: int, parameter=Parameter('beta')) -> QuantumCircuit:
    # TODO: use ancillas to reduce depth of mcp?
    num_qubits = int(np.ceil(np.log2(N+1))) * T
    state_prep_circuit = state_prep(N, T)
    mixer = QuantumCircuit(num_qubits)
    mixer.append(
        state_prep_circuit.inverse(),
        range(num_qubits)
    )
    # mixer.save_statevector('after_prep')
    mixer.x(-1)
    mixer.mcp(-parameter, list(range(num_qubits - 1)), -1, ctrl_state=0)
    mixer.x(-1)
    # mixer.save_statevector('after_phase')
    mixer.append(
        state_prep_circuit,
        range(num_qubits)
    )
    # mixer.save_statevector('after_unprep')
    return mixer

In [ ]:
mixer_circuit = get_mixer_operator(3, 4)

In [ ]:
mixer_circuit.decompose(['circuit*'], reps=2).draw(fold=-1)

In [ ]:
state_prep_circuit = state_prep(4,5)
state_prep_circuit.save_statevector()
state_prep_circuit.decompose(['circuit*'], reps=2).draw(fold=-1)

In [ ]:
sim = AerSimulator()


In [ ]:
tsp = transpile(state_prep_circuit, backend=sim)

In [ ]:
res = sim.run(tsp).result()

In [ ]:
res.results[0].data

In [ ]:
sv = res.data()["statevector"]
sv = np.real_if_close(sv)

In [ ]:
sv = sv[np.abs(sv) >= 10**-4]
np.nonzero(sv)

In [ ]:
svr = sv.reshape([8]*5)

In [ ]:
svr[np.abs(svr) <= 10**-4] = 0

In [ ]:
svr[3,4,3,1,:]

In [ ]:
import pickle

In [ ]:
with open('/lustre/scratch127/qpg/jc59/hubo/simulation.optimisation.default_test_N4_W5.cvar0.25.p2.shots1000.initramp.pkl', 'rb') as f:
    data = pickle.load(f)


In [ ]:
last_samples = data['history'][-1]
last_samples = last_samples[3]
keys = list(last_samples.keys())


In [ ]:
hamiltonian = data["hamiltonian"]

In [ ]:
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [ ]:
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian


In [ ]:
costs = evaluate_sparse_pauli_samples(keys, hamiltonian)

In [ ]:
np.argmax(costs)

In [ ]:
keys[50] # '01000100110011001100' -> 2, 2, 3, 3, 3 -> u1+,u1+,u1-,u1-,u1- : 4*10 + (2-0)**2 + (1-5)**2 + (1-0)**2 + (1-0)**2 

In [ ]:
costs[50]

In [ ]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N4_W5.gfa'
graph, n, N, T = gfa_file_to_graph(filepath, [2,1,1,1])


In [ ]:
new_hamiltonian = graph_to_hubo_hamiltonian(graph, n, N, T, 10)

In [ ]:
evaluate_sparse_pauli_samples(
    ['00011001'+'1111'*3, '10011111100111011001', '100000011000'+'1111'*2, '0000'+'0100'+'1010'+'0000'+'0110', '1110'+'1000'+'0010'+'1100'+'1000'],
    new_hamiltonian
)

In [ ]:
23 % 12

In [ ]:
x = list(range(12))
def even_swap(x):
    for i in range(0, len(x), 2):
        x[i], x[i+1] = x[i+1], x[i]
    return x

def odd_swap(x):
    for i in range(1, len(x)-1, 2):
        x[i], x[i+1] = x[i+1], x[i]
    return x

def middle_swap(x):
    for i in range(0, len(x), 4):
        x[i], x[i+1] = x[i+1], x[i]
    return x

print(x)
for j in range(2*len(x)):
    if j % 2 == 0:
        x = even_swap(x)
    else:
        x = odd_swap(x)
    print(x)
    if j % len(x) == len(x) - 1:
        print(j)
        print('MIDDLE SWAP')
        x = middle_swap(x)
        print(x)
    

In [ ]:
4 // 12

In [ ]:
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy

In [ ]:
ess = ExtendedSwapStrategy.from_line(list(range(20)), 20)

In [ ]:
nodes_list = [(0, 2, 5, 9), (3, 5, 13, 17), (1, 4, 5), (1, 4, 9),(0, 1, 3, 9), (6, 14, 18), (6, 10, 18), (6, 8, 10, 19), (0, 2, 6, 8, 10), (7, 9, 11, 13, 17), (0, 1, 4, 6, 12),]

In [ ]:
edge_map = {0: 16, 1: 14, 2: 18, 3: 19, 4: 10, 5: 8, 6: 12, 7: 6, 8: 0, 9: 1, 10: 2, 11: 4, 12: 5, 13: 3, 14: 7, 15: 9, 16: 13, 17: 15, 18: 11, 19: 17}

In [ ]:
for nodes in nodes_list:
    print(nodes)
    print(ess.distance_nodes(nodes))
    # mapped_nodes = tuple([edge_map[i] for i in nodes])
    # print(mapped_nodes)
    # print(ess.distance_nodes(mapped_nodes))
    print()

In [ ]:
initial_layout = [edge_map[i] for i in range(20)]
print(initial_layout)
new_layout = initial_layout
for j in range(20):
    new_layout = ess.apply_swap_layer(new_layout, j)
    print(new_layout)
print(new_layout)

In [2]:
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy
ess = ExtendedSwapStrategy.from_grid(4,5)

((0, 1), (2, 3), (6, 7), (8, 9), (10, 11), (12, 13), (16, 17), (18, 19))
((1, 2), (3, 4), (5, 6), (7, 8), (11, 12), (13, 14), (15, 16), (17, 18))
((0, 5), (10, 15), (1, 6), (11, 16), (2, 7), (12, 17), (3, 8), (13, 18), (4, 9), (14, 19))
((5, 10), (6, 11), (7, 12), (8, 13), (9, 14))
[((0, 1), (2, 3), (6, 7), (8, 9), (10, 11), (12, 13), (16, 17), (18, 19)), ((1, 2), (3, 4), (5, 6), (7, 8), (11, 12), (13, 14), (15, 16), (17, 18)), ((0, 1), (2, 3), (6, 7), (8, 9), (10, 11), (12, 13), (16, 17), (18, 19)), ((1, 2), (3, 4), (5, 6), (7, 8), (11, 12), (13, 14), (15, 16), (17, 18)), ((0, 5), (10, 15), (1, 6), (11, 16), (2, 7), (12, 17), (3, 8), (13, 18), (4, 9), (14, 19)), ((5, 10), (6, 11), (7, 12), (8, 13), (9, 14)), ((0, 1), (2, 3), (6, 7), (8, 9), (10, 11), (12, 13), (16, 17), (18, 19)), ((1, 2), (3, 4), (5, 6), (7, 8), (11, 12), (13, 14), (15, 16), (17, 18)), ((0, 1), (2, 3), (6, 7), (8, 9), (10, 11), (12, 13), (16, 17), (18, 19)), ((1, 2), (3, 4), (5, 6), (7, 8), (11, 12), (13, 14), (15, 1

In [3]:
len(ess)

12